# 🏆 [Day 33] 실전 GDS 복합 투영 & 신약 타겟팅 다차원 중심성 핸즈온 워크북

> **학습 목표**:
> 단순 타이핑 노동이 아닌, **'왜 이렇게 설계했는가?(Thinking Point)'**와 **'비교 대조군 실험(What-If Simulation)'**을 통해 GDS 토폴로지와 개인화 PageRank의 본질을 100% 체득합니다.
>
> 1. 🧐 **[생각하기 1]**: 왜 `Symptom`(증상)은 빼고 `Gene`(유전자)을 삼각망에 넣었는가?
> 2. ⚡ **[투영 & 3대 중심성]**: Degree(마당발) vs PageRank(실세) vs Betweenness(길목) 실측
> 3. 🎯 **[생각하기 2 & PPR]**: '유방암' 타겟팅 시 전역 1위와 어떻게 달라지는가? (PPR 표적 신약 도출)
> 4. 🔬 **[비교 실험 1]**: `BINDS`(유전자 결합)를 뺐을 때 신약 발굴력이 어떻게 무너지는가?
> 5. 💥 **[비교 실험 2]**: `UNDIRECTED`를 제거하고 단방향으로 돌렸을 때 점수가 왜 박살 나는가?

## 0. 환경 설정 및 Neo4j 연결

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv(".env")
load_dotenv("../.env")

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_cypher(query, **params):
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]

ver_res = run_cypher("RETURN gds.version() AS ver")[0]
print(f"✅ Neo4j GDS 플러그인 버전: {ver_res['ver']}")

## 🧐 [Thinking Point 1] 투영 전 질문: "왜 Symptom은 빼고 Gene은 넣었을까?"
- **`Symptom`(증상: 통증, 기침, 발열)**은 질병의 '결과'일 뿐, 치료의 타겟이 되지 못합니다.
- **`Gene`(유전자/단백질)**은 질병을 일으키는 생물학적 원인이자, 약물이 물리적으로 결합(`BINDS`)할 수 있는 **'열쇠 구멍(Target)'**입니다.
- 따라서 **`Disease(원인) ➔ Gene(표적) ➔ Compound(약물)`**의 삼각 메커니즘을 완성하기 위해 `Symptom`은 서브그래프에서 제외하고 `Gene`을 선택합니다!

## 1. 다중 이종 노드·관계 삼각 투영 (gds.graph.project)
- 노드: `Disease`, `Gene`, `Compound` 3개 레이블
- 관계: `ASSOCIATES`, `BINDS`, `TREATS` 3개 관계를 모두 `UNDIRECTED`로 투영

In [ ]:
# 기존 투영 멱등성 해제
run_cypher("CALL gds.graph.drop('bioMasterGraph', false) YIELD graphName")

proj_cypher = """
CALL gds.graph.project(
    'bioMasterGraph',
    ['Disease', 'Gene', 'Compound'],
    {
        ASSOCIATES: {type: 'ASSOCIATES', orientation: 'UNDIRECTED'},
        BINDS: {type: 'BINDS', orientation: 'UNDIRECTED'},
        TREATS: {type: 'TREATS', orientation: 'UNDIRECTED'}
    }
)
YIELD graphName, nodeCount, relationshipCount, projectMillis
"""
res = run_cypher(proj_cypher)[0]
print(f"✅ 인메모리 삼각 투영 완료: {res['graphName']}")
print(f"  • 투영 노드 수: {res['nodeCount']:,}개 / 엣지 수: {res['relationshipCount']:,}건 (무방향 2배 검산)")
print(f"  • 투영 소요 시간: {res['projectMillis']} ms")

## 2. 3대 중심성 지표(Degree vs PageRank vs Betweenness) 교차 비교

In [ ]:
# 1. 차수 중심성 (Degree: 마당발)
deg_df = pd.DataFrame(run_cypher("""
CALL gds.degree.stream('bioMasterGraph')
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
RETURN n.name AS name, labels(n)[0] AS type, toInteger(score) AS degree
ORDER BY degree DESC LIMIT 5
"""))

# 2. PageRank 중심성 (실세/권력자)
pr_df = pd.DataFrame(run_cypher("""
CALL gds.pageRank.stream('bioMasterGraph', {maxIterations: 20, dampingFactor: 0.85})
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
RETURN n.name AS name, labels(n)[0] AS type, round(score, 4) AS pagerank
ORDER BY pagerank DESC LIMIT 5
"""))

# 3. 매개 중심성 (Betweenness: 길목/브로커)
btw_df = pd.DataFrame(run_cypher("""
CALL gds.betweenness.stream('bioMasterGraph')
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
RETURN n.name AS name, labels(n)[0] AS type, round(score, 2) AS betweenness
ORDER BY betweenness DESC LIMIT 5
"""))

print("📊 [1. Degree Top 5 - 단순 연결 마당발]")
display(deg_df)
print("\n🌐 [2. PageRank Top 5 - 전역 영향력 실세]")
display(pr_df)
print("\n🌉 [3. Betweenness Top 5 - 네트워크 핵심 길목]")
display(btw_df)

## 🧐 [Thinking Point 2] PPR 전 예측: "유방암을 출발점으로 잡으면 전역 1위와 어떻게 달라질까?"
- **전역 PageRank 1위**: 아스피린, 포도당, 알코올처럼 세상 모든 생체 반응에 두루 쓰이는 '흔한 물질'이 1등을 차지합니다.
- **개인화 PageRank(PPR) 1위**: 전 세계 1등이 아니라, **'유방암-유전자-약물 연결망'을 집중적으로 타고 들어간 '타목시펜(Tamoxifen), 독소루비신(Doxorubicin)' 같은 실제 유방암 표적 치료제**가 압도적 1위로 올라옵니다!

## 3. 유방암(`breast cancer`) 타겟 개인화 PageRank (PPR) 신약 발굴

In [ ]:
target_disease = 'breast cancer'

ppr_res = run_cypher("""
MATCH (d:Disease {name: $disease_name})
WITH collect(id(d)) AS sources
CALL gds.pageRank.stream('bioMasterGraph', {
    maxIterations: 20,
    dampingFactor: 0.85,
    sourceNodes: sources
})
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
WHERE n:Compound
RETURN n.name AS compound_name, round(score, 6) AS ppr_score
ORDER BY ppr_score DESC
LIMIT 10
""", disease_name=target_disease)

df_ppr = pd.DataFrame(ppr_res)
print(f"🎯 [{target_disease} 타겟 개인화 PageRank Top 10 약물 후보]")
display(df_ppr)

## 🔬 [비교 실험 1] `BINDS`(유전자 결합)를 빼고 `TREATS`만 남기면 어떻게 될까?
- **목적**: 유전자 매개 정보(`BINDS`)가 없을 때 신약 발굴이 얼마나 빈약해지는지 실증합니다.

In [ ]:
# BINDS(유전자 결합)를 뺀 축소 투영 생성
run_cypher("CALL gds.graph.drop('treatsOnlyGraph', false) YIELD graphName")
run_cypher("""
CALL gds.graph.project(
    'treatsOnlyGraph',
    ['Disease', 'Compound'],
    {TREATS: {type: 'TREATS', orientation: 'UNDIRECTED'}}
)
""")

ppr_treats_only = run_cypher("""
MATCH (d:Disease {name: 'breast cancer'})
WITH collect(id(d)) AS sources
CALL gds.pageRank.stream('treatsOnlyGraph', {
    maxIterations: 20,
    dampingFactor: 0.85,
    sourceNodes: sources
})
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
WHERE n:Compound
RETURN n.name AS compound_name, round(score, 6) AS ppr_score
ORDER BY ppr_score DESC
LIMIT 5
""")

print("📉 [유전자 결합(BINDS) 제외 시 PPR 결과 - 알려진 기치료제만 나오고 신약 표적 발굴 불가]")
display(pd.DataFrame(ppr_treats_only))
run_cypher("CALL gds.graph.drop('treatsOnlyGraph') YIELD graphName")

## 💥 [비교 실험 2] `UNDIRECTED`를 제거하고 단방향(`NATURAL`)으로 돌리면 어떻게 될까?
- **목적**: 단방향일 때 왜 유방암에서 약물로의 경로가 끊겨 점수가 0점이 되는지 직접 검증합니다.

In [ ]:
# 단방향 투영 생성
run_cypher("CALL gds.graph.drop('directedGraph', false) YIELD graphName")
run_cypher("""
CALL gds.graph.project(
    'directedGraph',
    ['Disease', 'Gene', 'Compound'],
    ['ASSOCIATES', 'BINDS', 'TREATS']  // orientation을 안 주면 단방향(NATURAL)으로 저장됨
)
""")

directed_ppr = run_cypher("""
MATCH (d:Disease {name: 'breast cancer'})
WITH collect(id(d)) AS sources
CALL gds.pageRank.stream('directedGraph', {
    maxIterations: 20,
    dampingFactor: 0.85,
    sourceNodes: sources
})
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
WHERE n:Compound
RETURN n.name AS compound_name, round(score, 6) AS ppr_score
ORDER BY ppr_score DESC
LIMIT 5
""")

print("💥 [단방향(NATURAL) 적용 시 PPR 결과 - 경로가 역주행 불가로 끊겨 점수가 죄다 0점/동점으로 파탄남!]")
display(pd.DataFrame(directed_ppr))
run_cypher("CALL gds.graph.drop('directedGraph') YIELD graphName")

## 4. 메인 인메모리 투영 메모리 안전 해제 (Clean Drop)

In [ ]:
drop_res = run_cypher("CALL gds.graph.drop('bioMasterGraph') YIELD graphName")[0]
print(f"✅ 메인 투영 메모리 정상 반환 완료: {drop_res['graphName']}")